# Chapter 19: Field Development Optimization and VFP Tables

This notebook demonstrates the generation of Vertical Flow Performance (VFP) tables
using NeqSim pipe flow models. Topics include:
- Building a wellbore model with `PipeBeggsAndBrills`
- Sweeping wellhead pressure and flow rate to calculate BHP
- Generating VFP curves (BHP vs flow rate at different WHPs)
- Discussion of reservoir simulation coupling

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 19.1 Wellbore Flow Model Setup

We model a vertical production well using `PipeBeggsAndBrills`. The Beggs and Brill
correlation handles multiphase flow with gravitational and frictional pressure drops.
The tubing is modeled as a vertical pipe from reservoir depth to surface.

In [2]:
from neqsim import jneqsim

# Reservoir fluid — gas condensate
fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 90.0, 250.0)
fluid.addComponent("nitrogen", 0.01)
fluid.addComponent("CO2", 0.02)
fluid.addComponent("methane", 0.75)
fluid.addComponent("ethane", 0.08)
fluid.addComponent("propane", 0.05)
fluid.addComponent("n-butane", 0.03)
fluid.addComponent("n-pentane", 0.02)
fluid.addComponent("n-hexane", 0.02)
fluid.addComponent("n-heptane", 0.01)
fluid.addComponent("water", 0.01)
fluid.setMixingRule("classic")
fluid.setMultiPhaseCheck(True)

# Well parameters
well_depth_m = 3000.0
tubing_id_m = 0.1  # ~4 inch tubing ID
roughness_m = 2.5e-5

print(f"Well depth: {well_depth_m:.0f} m")
print(f"Tubing ID: {tubing_id_m*1000:.0f} mm")
print(f"Reservoir temperature: 90 C")

Well depth: 3000 m
Tubing ID: 100 mm
Reservoir temperature: 90 C


## 19.2 Single Point Wellbore Calculation

Run a single wellbore flow calculation to verify the model. We set the wellhead
conditions (pressure and temperature) and flow rate, then calculate the bottomhole
pressure (BHP) using `PipeBeggsAndBrills`.

In [3]:
# Wellhead conditions
whp_bara = 80.0
wht_C = 40.0
flow_rate_kg_hr = 30000.0

wellhead_stream = jneqsim.process.equipment.stream.Stream("Wellhead", fluid)
wellhead_stream.setFlowRate(flow_rate_kg_hr, "kg/hr")
wellhead_stream.setTemperature(wht_C, "C")
wellhead_stream.setPressure(whp_bara, "bara")

# Wellbore (pipe going downward = negative elevation from wellhead to BH)
# We model from wellhead to bottomhole: elevation = -well_depth (going down)
wellbore = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills("Wellbore", wellhead_stream)
wellbore.setPipeWallRoughness(roughness_m)
wellbore.setLength(well_depth_m)
wellbore.setElevation(-well_depth_m)  # downward from wellhead to reservoir
wellbore.setDiameter(tubing_id_m)

well_process = jneqsim.process.processmodel.ProcessSystem()
well_process.add(wellhead_stream)
well_process.add(wellbore)
well_process.run()

bhp = wellbore.getOutletStream().getPressure("bara")
bht = wellbore.getOutletStream().getTemperature("C")

print(f"Wellhead pressure:  {whp_bara:.1f} bara")
print(f"Bottomhole pressure: {bhp:.1f} bara")
print(f"Bottomhole temperature: {bht:.1f} C")
print(f"Pressure drop: {bhp - whp_bara:.1f} bar")

Wellhead pressure:  80.0 bara
Bottomhole pressure: 84.1 bara
Bottomhole temperature: 40.0 C
Pressure drop: 4.1 bar


## 19.3 VFP Table Generation

A VFP table maps (WHP, flow rate) → BHP. We sweep multiple wellhead pressures and
flow rates to build the complete performance surface.

In [4]:
# Define sweep ranges
whp_values = [40.0, 60.0, 80.0, 100.0, 120.0]  # bara
flow_values = np.linspace(5000, 60000, 12)  # kg/hr

# VFP table: dict of {WHP: [BHP at each flow rate]}
vfp_table = {}

for whp in whp_values:
    bhp_list = []
    for flow in flow_values:
        wellhead_stream.setFlowRate(float(flow), "kg/hr")
        wellhead_stream.setPressure(float(whp), "bara")
        wellhead_stream.setTemperature(wht_C, "C")
        try:
            well_process.run()
            bhp_val = wellbore.getOutletStream().getPressure("bara")
            bhp_list.append(bhp_val)
        except Exception as e:
            bhp_list.append(float('nan'))
    vfp_table[whp] = bhp_list

# Print a preview
print(f"{'Flow (t/hr)':<15}", end="")
for whp in whp_values:
    print(f"WHP={whp:.0f}", end="    ")
print()
print("-" * 80)
for i, flow in enumerate(flow_values):
    print(f"{flow/1000:<15.1f}", end="")
    for whp in whp_values:
        bhp_val = vfp_table[whp][i]
        print(f"{bhp_val:<12.1f}", end="")
    print()

Flow (t/hr)    WHP=40    WHP=60    WHP=80    WHP=100    WHP=120    
--------------------------------------------------------------------------------
5.0            68.0        101.6       134.5       164.1       190.2       
10.0           56.9        92.0        125.0       156.5       184.9       
15.0           41.3        84.0        119.4       151.7       180.8       
20.0           0.6         71.4        111.4       145.8       176.3       
25.0           0.6         51.2        99.9        138.0       170.2       
30.0           0.6         10.8        84.1        127.8       162.8       
35.0           0.6         10.8        61.6        114.6       153.5       
40.0           0.6         10.8        22.9        97.4        142.1       
45.0           0.6         10.8        22.9        73.4        127.7       
50.0           0.6         10.8        22.9        36.7        109.4       
55.0           0.6         10.8        22.9        36.7        85.3        
60.0           

## 19.4 VFP Curves Plot

Plot BHP vs flow rate at each wellhead pressure. These curves are the standard
VFP representation used in reservoir simulation coupling.

In [5]:
fig, ax = plt.subplots(figsize=(10, 7))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(whp_values)))

for idx, whp in enumerate(whp_values):
    bhps = np.array(vfp_table[whp])
    valid = ~np.isnan(bhps)
    ax.plot(flow_values[valid]/1000, bhps[valid], 'o-', color=colors[idx],
            linewidth=2, markersize=5, label=f'WHP = {whp:.0f} bara')

ax.set_xlabel('Flow Rate (t/hr)', fontsize=13)
ax.set_ylabel('Bottomhole Pressure (bara)', fontsize=13)
ax.set_title('VFP Curves: Bottomhole Pressure vs Flow Rate', fontsize=14)
ax.legend(title='Wellhead Pressure', fontsize=10, title_fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ch19_vfp_curves.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_42844\3213182066.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 19.5 VFP Surface Contour

A contour plot gives a more complete view of the BHP surface as a function of
both WHP and flow rate.

In [6]:
# Build 2D arrays for contour
WHP_grid, FLOW_grid = np.meshgrid(whp_values, flow_values / 1000)
BHP_grid = np.zeros_like(WHP_grid)

for j, whp in enumerate(whp_values):
    for i in range(len(flow_values)):
        BHP_grid[i, j] = vfp_table[whp][i]

fig, ax = plt.subplots(figsize=(10, 6))
cs = ax.contourf(FLOW_grid, WHP_grid, BHP_grid, levels=15, cmap='viridis')
cbar = plt.colorbar(cs, ax=ax, label='BHP (bara)')
ax.set_xlabel('Flow Rate (t/hr)', fontsize=12)
ax.set_ylabel('Wellhead Pressure (bara)', fontsize=12)
ax.set_title('VFP Surface: Bottomhole Pressure Contour', fontsize=13)

plt.tight_layout()
plt.savefig("../figures/ch19_vfp_surface.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_42844\3286534752.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 19.6 Reservoir–Wellbore Coupling Concept

In a coupled reservoir simulation, the VFP table is pre-computed and passed to the
reservoir simulator (e.g., Eclipse, OPM). At each timestep, the simulator:

1. Knows the reservoir pressure $P_r$ and desired rate $q$
2. Looks up BHP from the VFP table: $P_{bh} = \text{VFP}(q, P_{wh})$
3. Checks if $P_r > P_{bh}$ (flow is possible) or adjusts rate

This decouples the expensive multiphase flow calculation from the reservoir timestep.

**NeqSim's role** in this workflow:
- Generate VFP tables with thermodynamically consistent fluid properties
- Account for compositional effects (condensate dropout in tubing)
- Include accurate multiphase flow correlations (Beggs & Brill, OLGA-like)

The VFP tables can be exported in Eclipse VFPPROD keyword format for direct use.

## Summary

Key concepts:

1. **VFP tables** map (WHP, flow rate) → BHP using multiphase pipe flow models
2. **`PipeBeggsAndBrills`** handles gravitational and frictional pressure drops in vertical/inclined wells
3. The VFP surface shows how BHP increases with both flow rate (friction) and WHP (back-pressure)
4. VFP tables are **pre-computed** for efficiency in reservoir simulation coupling
5. NeqSim provides compositionally consistent VFP generation with proper thermodynamics